In [3]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import seaborn as sns

# -------------------------------------------------------------------------
# 1) Define Your Modified Inception Model
#    (Matches the code you used during training, with final 2-class layer)
# -------------------------------------------------------------------------
class InceptionV3Modified(nn.Module):
    def __init__(self):
        super(InceptionV3Modified, self).__init__()
        self.inception = models.inception_v3(pretrained=True)
        # Replace final FC layer for 2 classes
        in_features = self.inception.fc.in_features
        self.inception.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 2)
        )

    def forward(self, x):
        # In training mode, Inception v3 returns (aux_logits, main_logits).
        # In eval mode, it returns just main_logits.
        # If you need the auxiliary output at training time, adapt as needed.
        return self.inception(x)

# -------------------------------------------------------------------------
# 2) Load the Trained Checkpoint
# -------------------------------------------------------------------------
# Update with your checkpoint path:
checkpoint_path = "/content/drive/My Drive/FYP/InceptionModel/HQ2ltst_20241121/final_model_inception_20241121.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model and load weights
model = InceptionV3Modified().to(device)
model.eval()

if not os.path.exists(checkpoint_path):
    print(f"ERROR: The checkpoint file was not found at:\n  {checkpoint_path}")
    exit()
else:
    # If your checkpoint only contains the model’s state_dict:
    #   state_dict = torch.load(checkpoint_path, map_location=device)
    #   model.load_state_dict(state_dict)
    #
    # Otherwise, if your checkpoint is a dictionary, you might do:
    #   checkpoint = torch.load(checkpoint_path, map_location=device)
    #   model.load_state_dict(checkpoint["model_state_dict"])

    # Adjust based on how you saved your checkpoint:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    print(f"Successfully loaded checkpoint from {checkpoint_path}")

# -------------------------------------------------------------------------
# 3) Define Class Names and Transforms
#    (Ensure folder names match these class names exactly!)
# -------------------------------------------------------------------------
class_names = ["Glaucous_Winged_Gull", "Slaty_Backed_Gull"]

# Inception v3 expects 299×299 input
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

def preprocess_image(image_path):
    """
    Load an image, apply transforms, and return a 4D tensor (1, 3, 299, 299).
    """
    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0).to(device)

# -------------------------------------------------------------------------
# 4) Grad-CAM Implementation for Inception
#    (Use the last convolutional layer: model.inception.Mixed_7c)
# -------------------------------------------------------------------------
def generate_gradcam(model, image_tensor, target_layer):
    """
    Generates a Grad-CAM heatmap from the specified layer (e.g., Mixed_7c).
    Returns: (cam, predicted_class_idx, predicted_confidence).
    """
    model.eval()
    # Containers for the forward feature maps and gradients
    features = []
    grads = []

    # Hooks
    def forward_hook(module, input, output):
        features.append(output)

    def backward_hook(module, grad_in, grad_out):
        grads.append(grad_out[0])

    # Register hooks
    f_handle = target_layer.register_forward_hook(forward_hook)
    b_handle = target_layer.register_backward_hook(backward_hook)

    # Forward pass
    outputs = model(image_tensor)
    probs = F.softmax(outputs, dim=1)
    predicted_class_idx = torch.argmax(probs, dim=1).item()
    predicted_confidence = probs[0, predicted_class_idx].item()

    # Backward pass
    model.zero_grad()
    score = outputs[0, predicted_class_idx]
    score.backward()

    # Extract data
    gradient = grads[0].detach().cpu().numpy()[0]      # shape: (C, H, W)
    feature_map = features[0].detach().cpu().numpy()[0]  # shape: (C, H, W)

    # Remove hooks
    f_handle.remove()
    b_handle.remove()

    # Compute weights by global-average pooling the gradients
    weights = np.mean(gradient, axis=(1, 2))  # shape: (C,)

    # Weighted sum of feature maps
    cam = np.zeros(feature_map.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * feature_map[i, :, :]

    # ReLU and normalize
    cam = np.maximum(cam, 0)
    h, w = image_tensor.shape[2], image_tensor.shape[3]  # 299, 299
    cam = cv2.resize(cam, (w, h))
    cam -= cam.min()
    cam /= (cam.max() + 1e-8)

    return cam, predicted_class_idx, predicted_confidence

# -------------------------------------------------------------------------
# 5) Utility: Save Overlay Visualization & CSV
# -------------------------------------------------------------------------
def save_visualization(image_path, cam, predicted_idx, confidence, true_class, save_path):
    """
    Overlay the Grad-CAM heatmap onto the original image, and label it with:
    - True class
    - Predicted class
    - Confidence
    - Correct/Wrong
    """
    original_img = Image.open(image_path).resize((299, 299), Image.LANCZOS)
    original_np = np.array(original_img)

    # Heatmap
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Blend heatmap with original
    overlay = np.clip(0.5 * original_np + 0.5 * heatmap, 0, 255).astype(np.uint8)

    pred_label = class_names[predicted_idx]
    is_correct = (pred_label == true_class)
    confidence_text = f"{confidence*100:.2f}%"
    result_text = (
        f"True: {true_class} | Pred: {pred_label} | "
        f"Conf: {confidence_text} | {'Correct' if is_correct else 'Wrong'}"
    )

    # Annotate image
    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.putText(
        overlay_bgr,
        result_text,
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0) if is_correct else (0, 0, 255),
        2,
        cv2.LINE_AA
    )
    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
    cv2.imwrite(save_path, cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))

def save_statistics_csv(stat_list, csv_path):
    df = pd.DataFrame(stat_list)
    df.to_csv(csv_path, index=False)
    print(f"Statistics saved to {csv_path}")

# -------------------------------------------------------------------------
# 6) Process Dataset & Generate Grad-CAM
# -------------------------------------------------------------------------
def process_dataset(dataset_path, output_base, delete_wrong=True):
    """
    - dataset_path: must contain subfolders named exactly as in class_names.
    - For each image in each subfolder:
      1) Preprocess + Grad-CAM
      2) Save overlay in correct/wrong subfolder
      3) Optionally delete original if prediction is wrong
    - Save CSV with predictions
    - Save confusion matrix plot
    """
    dataset_path = Path(dataset_path)
    if not dataset_path.exists():
        print(f"ERROR: Dataset path not found: {dataset_path}")
        return

    # Output folder with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path(output_base) / timestamp
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Results will be saved to: {output_dir}")

    stats_list = []

    # Process each subfolder
    for subfolder in sorted(dataset_path.iterdir()):
        if not subfolder.is_dir():
            continue
        true_class = subfolder.name
        if true_class not in class_names:
            print(f"Skipping folder '{true_class}' as it is not in class_names.")
            continue

        # Create subfolders for correct & wrong
        out_class_dir = output_dir / true_class
        correct_dir = out_class_dir / "correct"
        wrong_dir = out_class_dir / "wrong"
        correct_dir.mkdir(parents=True, exist_ok=True)
        wrong_dir.mkdir(parents=True, exist_ok=True)

        image_files = list(subfolder.glob("*.jpg")) \
                    + list(subfolder.glob("*.jpeg")) \
                    + list(subfolder.glob("*.png"))
        print(f"Processing {len(image_files)} images in '{true_class}'...")

        # Target layer is the last convolutional layer in Inception v3
        # This is typically Mixed_7c
        target_layer = model.inception.Mixed_7c

        for image_path in tqdm(image_files, desc=f"Processing {true_class}"):
            try:
                # Preprocess
                image_tensor = preprocess_image(image_path)

                # Grad-CAM
                cam, pred_idx, pred_conf = generate_gradcam(model, image_tensor, target_layer)
                pred_label = class_names[pred_idx]
                is_correct = (pred_label == true_class)

                # Where to save
                save_subdir = correct_dir if is_correct else wrong_dir
                save_filename = f"{image_path.stem}_gradcam.png"
                save_path = str(save_subdir / save_filename)

                # Save overlay
                save_visualization(str(image_path), cam, pred_idx, pred_conf, true_class, save_path)

                # Update stats
                stats_list.append({
                    "file": str(image_path),
                    "true_class": true_class,
                    "predicted_class": pred_label,
                    "confidence": pred_conf,
                    "correct": is_correct,
                    "output_file": save_path
                })

                # Optionally delete wrong images
                if not is_correct and delete_wrong:
                    os.remove(str(image_path))

            except Exception as e:
                print(f"Error processing {image_path}: {e}")

    # Save stats
    csv_path = output_dir / "statistics.csv"
    save_statistics_csv(stats_list, csv_path)

    # Confusion matrix
    try:
        df = pd.DataFrame(stats_list)
        if not df.empty:
            confusion = pd.crosstab(
                df["true_class"], df["predicted_class"],
                rownames=["True"], colnames=["Predicted"]
            )
            plt.figure(figsize=(6, 4))
            sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues")
            plt.title("Confusion Matrix")
            cm_path = output_dir / "confusion_matrix.png"
            plt.savefig(cm_path, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Confusion matrix saved to: {cm_path}")
    except Exception as e:
        print(f"Error generating confusion matrix: {e}")

# -------------------------------------------------------------------------
# 7) Main Execution
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # Folder containing two subfolders (each named like "Glaucous_Winged_Gull" or "Slaty_Backed_Gull")
    dataset_path = "/content/drive/My Drive/FYP/Black Background/"

    # Where to save Grad-CAM outputs and logs
    output_base = "/content/drive/My Drive/FYP/InceptionAnalysis"

    # If True, wrong predictions are permanently deleted from 'dataset_path'
    process_dataset(dataset_path, output_base, delete_wrong=False)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
100%|██████████| 104M/104M [00:01<00:00, 109MB/s] 
<ipython-input-3-8dc93ce4f012>:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle m

Successfully loaded checkpoint from /content/drive/My Drive/FYP/InceptionModel/HQ2ltst_20241121/final_model_inception_20241121.pth
Results will be saved to: /content/drive/My Drive/FYP/InceptionAnalysis/20250218_130253
Processing 92 images in 'Glaucous_Winged_Gull'...


Processing Glaucous_Winged_Gull:   0%|          | 0/92 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1827: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
Processing Glaucous_Winged_Gull: 100%|██████████| 92/92 [02:22<00:00,  1.54s/it]


Processing 131 images in 'Slaty_Backed_Gull'...


Processing Slaty_Backed_Gull: 100%|██████████| 131/131 [04:07<00:00,  1.89s/it]


Statistics saved to /content/drive/My Drive/FYP/InceptionAnalysis/20250218_130253/statistics.csv
Confusion matrix saved to: /content/drive/My Drive/FYP/InceptionAnalysis/20250218_130253/confusion_matrix.png
